# The Transformer Block

## Important Information

- Email: [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
- Office Hours take place in Duke 209, [Office Hours Schedule](https://joannabieri.com/schedule.html)
- [Class Website](https://joannabieri.com/machine_learning.html)
- [Syllabus](https://joannabieri.com/machinelearning/IntroMachineLearning.pdf)

:::{.callout-important icon=false}
## How to use these notes

Two kinds of box show up in these notes.

**Blue Q boxes** are questions for you to answer **by hand, in a notebook, with a pen.** Not because I am old fashioned. Writing something down by hand is slow, and slow is the point: it is very hard to write an explanation you do not actually understand. You are welcome to use AI in this class for the mechanics of code, but these boxes are the part where you do the thinking yourself. Bring your written notes to class, I will ask to see them.

**Green You Try boxes** are optional code for you to work through. Nothing is collected and nothing is graded. They are there because you will learn more from changing a number and rerunning than from watching me do it.

**Every code cell begins with a tag** that tells you what to do with it.

- `# RUN THIS.` Setup, loading data, a plot. Copy it, run it, move on. You do not need to be able to write it from memory.
- `# LEARN TO WRITE THIS.` The pattern of the day. The homework will ask you for it, and so will the exam. Type it out yourself at least once rather than pasting it.
- `# DEMO ONLY.` Code that exists to show one idea. You do not need to be able to write it yourself.
- `# UNDERSTAND THESE STEPS.` Code that shows how something works inside, one small step at a time. Read it and make sure you understand what each line does. You do not need to memorize it or be able to write it from scratch.
:::

**Reading:** Geron chapter 15, inside "Attention Is All You Need: The Original Transformer Architecture", the sections **Positional encodings**, **Multi-Head Attention (MHA)** and **Building the rest of the Transformer**.

We already built attention by hand. Every word makes a **query**, a **key** and a **value**. We compare each query to every key, softmax turns those scores into weights that add up to 1, and each word's output is the weighted average of the values:

$$\text{attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d}}\right) V$$

Remember "the pizza was cold": the query for "cold" asked "what thing am I describing?", the key for "pizza" said "I am a thing", so "cold" put most of its weight on "pizza" and its new vector became mostly about food.

What we built by hand is called one **attention head**: one set of queries, keys and values, making one table of weights. Today we put it inside a **transformer block**, which is what every modern language model is made of. The good news is that a transformer block only has four pieces, and you have seen most of them already!

1. **Multi-head attention**: several attention heads, side by side.
2. **Positional information**: so the network knows the order of the words.
3. **Residual connections and layer norm**: Instead of only passing on what attention made, we add it to what came in: X + attention(X) and we add a version of batch norm.
4. **A feed-forward network**: a small MLP.

Here is the plan for today:

1. Make some reviews where the **order** of the words decides the answer.
2. Look at multi-head attention with `nn.MultiheadAttention`.
3. See that attention, on its own, can not tell what order the words came in, and fix it.
4. Add the residual connections, layer norm and feed-forward, and write one block as a class.
5. Train it on the reviews, with and without positions.

# The Data - Reviews Where Order Matters

We are going to use the made up movie reviews from the RNN notes again, but with one change. Every review now has **two** feeling words, one positive and one negative, at random spots. **The last feeling word decides.** "boring at the start ... great by the ending" is positive, and "great at the start ... boring by the ending" is negative.

Both kinds of review have exactly the same words in them, so the ONLY way to get these right is to know which feeling word came first.

Why new reviews? When we built attention from scratch, every review had just one feeling word, so attention could get 1.0 without ever knowing where a word was. Here the order is the whole point, so we will see right away whether our model knows the order or not.

In [ ]:
# RUN THIS. Imports, the words, and a function that makes reviews with two feeling words.
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

feeling_words = ["great", "good", "fun", "loved", "wonderful",       # positive, ids 0 to 4
                 "bad", "boring", "awful", "hated", "terrible"]      # negative, ids 5 to 9
filler_words = ["the", "movie", "was", "and", "a", "plot", "actors", "it", "story", "long",
                "music", "ending", "i", "saw", "with", "my", "friends", "on", "friday", "popcorn"]
vocab = feeling_words + filler_words      # every word gets an id: its spot in this list
vocab_size = len(vocab)

def make_reviews(n_reviews, n_words, seed):
    generator = torch.Generator().manual_seed(seed)       # so we get the same reviews every time
    # Start with all filler words (ids 10 to 29)
    X = torch.randint(10, vocab_size, (n_reviews, n_words), generator=generator)
    # One positive word (ids 0 to 4) and one negative word (ids 5 to 9) for every review
    positive = torch.randint(0, 5, (n_reviews,), generator=generator)
    negative = torch.randint(5, 10, (n_reviews,), generator=generator)
    # Two different spots: the second is the first moved over by 1 to n_words - 1 places
    spot_pos = torch.randint(0, n_words, (n_reviews,), generator=generator)
    spot_neg = (spot_pos + torch.randint(1, n_words, (n_reviews,), generator=generator)) % n_words
    X[torch.arange(n_reviews), spot_pos] = positive
    X[torch.arange(n_reviews), spot_neg] = negative
    # Label 1 (positive) if the positive word comes LAST
    y = (spot_pos > spot_neg).long()
    return X, y

def to_words(review):
    words = []
    for word_id in review:
        words.append(vocab[word_id])
    return " ".join(words)

print("vocabulary size:", vocab_size)

In [ ]:
# RUN THIS. 4,000 reviews to train on and 1,000 to validate on, 20 words each.
X_train, y_train = make_reviews(4000, n_words=20, seed=0)
X_valid, y_valid = make_reviews(1000, n_words=20, seed=1)

print("X_train:", X_train.shape)
print("share of positive reviews in validation:", round(y_valid.float().mean().item(), 3))
for i in range(4):
    print(y_train[i].item(), " ", to_words(X_train[i]))

Read a few of them and find the two feeling words. The label is 1 when the positive word comes last.

---

# Several Heads Side by Side - Multi-Head Attention

One attention head makes one table of weights: for every word, how much it looks at every other word. But a word might need to look at different things for different reasons. In "my friends loved the movie", "loved" might want to know **who** loved it and also **what** they loved. With one head, "loved" has only one query, so it can only ask one question. With two heads it gets two queries: head 1 can ask "who did the loving?" and find "friends", while head 2 asks "what was loved?" and finds "movie".

**Multi-head attention** runs several attention heads at the same time, side by side. Each head has its own queries, keys and values, so it makes its own table of weights. Then the outputs of all the heads are put side by side and one `nn.Linear` mixes them back together.

![Two heads, side by side. Each head gets its own queries, keys and values, and works with 8 of the 16 numbers.](images/01-multihead.png)

With 16 numbers per word and 2 heads, each head works with $16 / 2 = 8$ numbers. The two outputs of 8 numbers go side by side and make 16 again:

$$\text{output} = \big[\,\text{head}_1 \;\; \text{head}_2\,\big]\, W_O$$

Why split the 16 numbers between the heads instead of giving each head all 16? Each head only needs enough numbers to ask its own question, and splitting means two heads cost the same number of weights as one big head. Why the extra `nn.Linear` ($W_O$) at the end? The two heads found different information, and this layer learns how to mix it into one list of 16 numbers for the next layer.

So the same 16 numbers go in and 16 numbers come out for every word. In PyTorch this is `nn.MultiheadAttention`.

In [ ]:
# UNDERSTAND THESE STEPS. Multi-head attention on "loved the popcorn".
torch.manual_seed(42)
embed = nn.Embedding(vocab_size, 16)              # every word becomes 16 numbers
review = torch.tensor([[3, 10, 29]])              # 1 review: "loved the popcorn"
vectors = embed(review)
print("vectors:", vectors.shape)                  # (reviews, words, numbers)

attention = nn.MultiheadAttention(
    embed_dim=16,          # 16 numbers per word go in and come out
    num_heads=2,           # 2 heads, so each head works with 16 / 2 = 8 numbers
    batch_first=True)      # our data is (reviews, words, numbers), with reviews first

# self-attention: the words are the queries, the keys AND the values, so we send vectors in 3 times
output, weights = attention(vectors, vectors, vectors,
                            average_attn_weights=False)   # give back every head's table, not the average
print("output: ", output.shape)
print("weights:", weights.shape)

In [ ]:
# UNDERSTAND THESE STEPS. Each head made its own 3 by 3 table of weights.
# torch.round(..., decimals=2) keeps 2 decimals, .detach() hides the grad_fn line in the printout.
print("head 1")
print(torch.round(weights[0, 0].detach(), decimals=2))
print("head 2")
print(torch.round(weights[0, 1].detach(), decimals=2))

Here is what `weights` looks like. Its shape is (1, 2, 3, 3): we sent in **1** review, there are **2** heads, and each head makes its own **3 by 3** table, one row and one column for each of the 3 words.

![The two tables of weights from the printout above.](images/04-head-tables.png)

Read a table one row at a time. The "loved" row is how much "loved" looks at "loved", "the" and "popcorn", and every row adds up to 1. The two heads have different tables because they have different weights. This layer is brand new, so the tables are random for now. They only become useful after training.

BEWARE: `nn.MultiheadAttention` gives back **two** things, the output and the weights, like `nn.RNN` did. **The output is what goes on to the next layer.**

:::{.callout-note icon=false}
## Q1. Write this one out by hand

**a.** Why use several attention heads instead of one? Use the words in "my friends loved the movie" to make up an example.

**b.** A word has 512 numbers and there are 8 heads (the sizes in the original transformer paper). How many numbers does each head work with? How many come out of `nn.MultiheadAttention` for each word?
:::

---

# Where Is Each Word? - Positional Information

Remember how when we trained a single attention model the model could not see the order of the words. Nothing in the plain attention model says where a word is. An RNN reads word 1, then word 2, and carries its hidden state along, so it knows the order. Attention sees all of the words at once. (Seeing every word at once is also what fixed the RNN's problem with long reviews: every word is one step away from every other word, so the gradient does not have to go back through 100 steps to reach the start of a review.)

Let's check this with multi-head attention too. Below we shuffle the three words, "popcorn loved the", and send them through the same attention layer.

In [ ]:
# DEMO ONLY. Shuffle the words and look at the output again.
# .detach() only hides the grad_fn line in the printout, it does not change the numbers.
shuffled = vectors[:, [2, 0, 1], :]          # "popcorn loved the"
output_shuffled, weights_shuffled = attention(shuffled, shuffled, shuffled)

print("output for 'loved' in order:   ", torch.round(output[0, 0, :4].detach(), decimals=4))
print("output for 'loved' shuffled:   ", torch.round(output_shuffled[0, 1, :4].detach(), decimals=4))

The output for "loved" is exactly the same (we printed the first 4 of its 16 numbers). Shuffling the words only shuffles the rows of the output. For our new reviews that is a big problem: "boring ... great" and "great ... boring" would look the same. We still need to fix the word order!

The fix is to give every **spot** in the review its own row of learned numbers. This is like learning an embeding that encodes the location of the word in the sentence. So every word has and embedding and every location in the sequence has an embedding so that two words that are the same but in different locations have different vectors going into the system. We just add together the two pieces to combine the data.

![The word numbers plus the spot numbers is what goes into the block.](images/02-positions.png)

This is called a **positional embedding**.

In [ ]:
# UNDERSTAND THESE STEPS. Add the numbers for each spot to the numbers for each word.
torch.manual_seed(42)
position_embed = nn.Embedding(
    num_embeddings=20,     # one row for every spot: reviews are at most 20 words long (a longer review would need more rows)
    embedding_dim=16)      # the same 16 numbers as the words, so we can add them

spots = torch.arange(3)                      # spots 0, 1, 2 for a 3 word review
print("spots:", spots)

vectors_with_spots = vectors + position_embed(spots)
print("vectors_with_spots:", vectors_with_spots.shape)

The shape did not change, it is still 16 numbers per word. We just now have encoded information about the spot (location) that we also get to learn during training!

:::{.callout-note icon=false}
## Q2. Write this one out by hand

**a.** Without positions, why would "great movie, boring ending" and "boring movie, great ending" look the same to attention?

**b.** Why do the spot numbers have to have the same size (16) as the word numbers?
:::

---

# The Rest of the Block - Residual Connections, Layer Norm, Feed-Forward

Attention lets the words look at each other. The full transformer block adds three more pieces around attention:

Each piece has one simple job:

1. **Residual connection**: remember who I am. Add what attention found to what came in.
2. **Layer norm**: don't get lost in the sauce. Keep each word's numbers a good size.
3. **Feed-forward**: self reflection. Each word works on its own numbers, one word at a time.

![The three extra pieces of the block, each with one job.](images/05-three-pieces.png)

Here is a little more about each one.

**Residual connection (the skip).** Instead of only passing on what attention made, we **add** it to what came in: $X + \text{attention}(X)$.

Here is why we want it. When we built attention by hand, "cold" started as [0, 1] (about temperature) and attention turned it into [0.711, 0.096], mostly about food. It lost most of its temperature! With the skip, the new "cold" is $[0, 1] + [0.711, 0.096] = [0.711, 1.096]$: it knows it is about temperature AND that the cold thing is food. If attention has nothing useful to add, the words still get through unchanged, and on the way back the gradient has a short path through the add. That is how you can stack dozens of these blocks without the gradient vanishing (remember vanishing gradients from backpropagation).

**Layer norm.** When we learned about optimizers, batch norm kept the numbers a good size by using the mean and standard deviation of a **batch**. Every skip layer adds more numbers on top of what came in, so over many blocks the numbers can keep growing, and big numbers make training unstable. Layer norm does the same thing as batch norm, but for **one word at a time**: it takes that word's 16 numbers, subtracts their mean and divides by their standard deviation. Then it multiplies and shifts by learned numbers, like batch norm does. It does not need the other reviews in the batch, so it works the same in training and in prediction.

In [ ]:
# MINI EXAMPLE. Layer norm on one word with 4 numbers.
norm = nn.LayerNorm(4)
word = torch.tensor([[2.0, 4.0, 6.0, 8.0]])
print(norm(word).detach())

The mean of 2, 4, 6, 8 is 5 and the standard deviation is $\sqrt{5} \approx 2.236$. So 2 becomes $(2 - 5)/2.236 = -1.3416$ and 8 becomes $(8 - 5)/2.236 = 1.3416$. The numbers are now centered on 0 and spread out by about 1, no matter how big they were before. The learned multiply and shift start out at 1 and 0, so a brand new layer norm does only the centering and scaling.

**Feed-forward.** Attention mixes information **between** words. The feed-forward part is a small MLP, `Linear`, `ReLU`, `Linear`, that works on **each word on its own**. It takes 16 numbers in, goes up to 32 in the middle, and comes back to 16. Going up to more numbers in the middle gives the network more room to combine what it learned (32 is a choice, the original paper went up 4 times). It has to come back to 16 so the shape out matches the shape in. So first each word gathers information from the other words (attention), and then the feed-forward part works on that new information, one word at a time.

Put it all together and one block is the two lines below. 

$$Z = \text{LayerNorm}\big(X + \text{MultiHeadAttention}(X)\big)$$
$$\text{out} = \text{LayerNorm}\big(Z + \text{FeedForward}(Z)\big)$$

What is the transformer doing:

* Sequence of embeddings (with position goes in) 
* We let the words look at each other = attention
* We add that attention result to what came in, and normalize 
* Then let each word think on its own = linear MLP 
* We add the linear result to what came from attention, and normalize again.

If I am a word going through the transformer this is what I am thinking!

   - Hooray, I am now a number and I have information about my position! (embedding + positional embedding)
   - Let's check in with the other words (multi-head attention)
   - But also remember who I am (residual connection, X + attention(X))
   - Let me do some self reflection now that I know myself and my neighbors so well (feed-forward, one word at a time)
   - But not get too lost in the sauce! (layer norm, keeps my numbers a good size)
   - *Rinse and repeat (loop through the blocks) until I'm ready to predict! (mean over the words + nn.Linear)*


![One transformer block. The red arrows are the residual connections.](images/03-block.png)

:::{.callout-note icon=false}
## Q3. Write this one out by hand

**a.** In $Z = \text{LayerNorm}(X + \text{attention}(X))$, what happens if attention gives back all zeros? Why is that a good thing in a network with many blocks?

**b.** Batch norm uses a batch of rows. What does layer norm use instead?
:::

:::{.callout-note icon=false}
## Q4. Write this one out by hand

**a.** Attention and the feed-forward network both change the numbers for each word. What is the difference in what they look at?

**b.** One block takes in 16 numbers per word and gives back 16 numbers per word. Why is it important that the shape out is the same as the shape in?
:::

---

# One Transformer Block in PyTorch

Here is the whole block as a class, the same way we wrote `ReviewNet` for the RNN. `__init__` makes the four layers and `forward` follows the two formulas above line by line.

In [ ]:
# LEARN TO WRITE THIS. One transformer block.
class TransformerBlock(nn.Module):

    def __init__(self, n_numbers=16, n_heads=2):
        super().__init__()                                   # always the first line
        self.attention = nn.MultiheadAttention(n_numbers, n_heads, batch_first=True)   # the heads, side by side
        self.norm1 = nn.LayerNorm(n_numbers)                 # layer norm after attention
        self.feed_forward = nn.Sequential(
            nn.Linear(n_numbers, 2 * n_numbers),     # up to 32 numbers
            nn.ReLU(),
            nn.Linear(2 * n_numbers, n_numbers))     # back down to 16
        self.norm2 = nn.LayerNorm(n_numbers)                 # layer norm after the feed-forward part

    def forward(self, X):
        attention_out, weights = self.attention(X, X, X)    # the words look at each other
        Z = self.norm1(X + attention_out)                    # skip, then layer norm
        out = self.norm2(Z + self.feed_forward(Z))           # feed-forward, skip, layer norm
        return out                                           # the same shape as X

In [ ]:
# UNDERSTAND THESE STEPS. 5 reviews of 20 words go in, the same shape comes out.
torch.manual_seed(42)
block = TransformerBlock()
X_fake = torch.randn(5, 20, 16)              # made up numbers: 5 reviews, 20 words, 16 numbers
print("going in: ", X_fake.shape)
print("coming out:", block(X_fake).shape)

n_block = 0
for p in block.parameters():
    n_block = n_block + p.numel()
print("numbers to learn in one block:", n_block)

Same shape in and out, so we could put another block right on top of this one, and another, and another.

# A Review Classifier With One Block

To classify a review we need a few more pieces around the block:

1. `nn.Embedding` turns each word into 16 numbers, and a second `nn.Embedding` gives each spot its 16 numbers. We add them.
2. The transformer block.
3. The block gives back 16 numbers for **every** word. We need one answer per review, so we take the **mean over the words** with `.mean(dim=1)`. (The RNN used its last memory instead.)
4. `nn.Linear(16, 2)` turns that into 2 scores, negative and positive.

The `use_positions` flag lets us turn the spot numbers off, so we can see what they do.

In [ ]:
# LEARN TO WRITE THIS. Embedding plus positions, one block, mean over the words, 2 scores.
class ReviewTransformer(nn.Module):

    def __init__(self, n_words, use_positions=True):
        super().__init__()
        self.use_positions = use_positions
        self.embed = nn.Embedding(vocab_size, 16)            # 16 numbers for each word
        self.position_embed = nn.Embedding(n_words, 16)      # 16 numbers for each spot
        self.block = TransformerBlock(16, n_heads=2)
        self.out = nn.Linear(16, 2)                          # 2 scores: negative, positive

    def forward(self, X):
        vectors = self.embed(X)                              # (reviews, words) -> (reviews, words, 16)
        if self.use_positions:
            spots = torch.arange(X.shape[1])                 # 0, 1, ..., n_words - 1
            vectors = vectors + self.position_embed(spots)
        Z = self.block(vectors)                              # (reviews, words, 16)
        summary = Z.mean(dim=1)                              # average over the words: (reviews, 16)
        scores = self.out(summary)
        return scores

In [ ]:
# RUN THIS. The same training loop we used for the RNN. It returns the validation accuracy after every epoch.
def train_model(model, X_train, y_train, X_valid, y_valid, n_epochs):
    torch.manual_seed(42)                         # so the batches come in the same order every time
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    valid_accuracy = []
    for epoch in range(n_epochs):
        model.train()                             # training mode
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()                 # clear the old gradients
            scores = model(X_batch)               # forward pass: 2 scores per review
            loss = loss_fn(scores, y_batch)       # how far off are we?
            loss.backward()                       # backward pass
            optimizer.step()                      # one step of Adam

        model.eval()                              # prediction mode
        with torch.no_grad():
            y_pred = model(X_valid).argmax(dim=1)     # the class with the biggest score
            accuracy = (y_pred == y_valid).float().mean().item()
        valid_accuracy.append(accuracy)
        print("epoch", epoch + 1, "  valid accuracy:", round(accuracy, 4))

    return valid_accuracy

Let's train it! First, without the spot numbers. What do you think will happen?

In [ ]:
# LEARN TO WRITE THIS. One block, NO positions.
torch.manual_seed(42)
model_no_spots = ReviewTransformer(n_words=20, use_positions=False)
accuracy_no_spots = train_model(model_no_spots, X_train, y_train, X_valid, y_valid, n_epochs=10)

It never learns: the validation accuracy stays between 0.461 and 0.526 for all 10 epochs. That is no better than flipping a coin. We can see why by swapping the two feeling words in one review, which flips its label, and asking the model about both.

In [ ]:
# DEMO ONLY. Swap the two feeling words in one review and ask the model about both.
review = X_valid[0].clone()
swapped = review.clone()
is_feeling = review < 10                           # True at the 2 spots with a feeling word
swapped[is_feeling] = review[is_feeling].flip(0)   # put the two feeling words in each other's spot

print("label", y_valid[0].item(), " ", to_words(review))
print("label", 1 - y_valid[0].item(), " ", to_words(swapped))

model_no_spots.eval()
with torch.no_grad():
    print("scores:", model_no_spots(review.reshape(1, 20)))
    print("scores:", model_no_spots(swapped.reshape(1, 20)))

Two reviews with opposite labels get exactly the same scores. Without positions the model can only see **which** words are in the review, never **where**, so it has no way to tell these apart. Now let's try the same model with the spot numbers turned on!

In [ ]:
# LEARN TO WRITE THIS. One block, WITH positions.
torch.manual_seed(42)
model_spots = ReviewTransformer(n_words=20, use_positions=True)
accuracy_spots = train_model(model_spots, X_train, y_train, X_valid, y_valid, n_epochs=10)

In [ ]:
# RUN THIS. The two learning curves.
epochs = range(1, 11)
plt.figure(figsize=(6, 4))
plt.plot(epochs, accuracy_no_spots, marker="o", label="no positions")
plt.plot(epochs, accuracy_spots, marker="o", label="with positions")
plt.xlabel("epoch")
plt.xticks(epochs)                   # only whole epochs on the x axis
plt.ylabel("validation accuracy")
plt.ylim(0.4, 1.05)
plt.legend()
plt.grid()
plt.show()

Look at that! With positions the same block gets to **0.956** after 10 epochs. The only thing we changed was adding 320 numbers to learn, 16 for each of the 20 spots ($20 \times 16 = 320$). Those numbers let the model see that "boring" in spot 3 and "great" in spot 17 is not the same review as "great" in spot 3 and "boring" in spot 17.

:::{.callout-note icon=false}
## Q5. Write this one out by hand

**a.** The model without positions got about 0.5 on these reviews. Would it also fail on the reviews we used for the RNN, where one feeling word decides and its spot does not matter? Why or why not?

**b.** Our model takes the mean over the words of the block's output. Why do we need one list of 16 numbers per review before the last `nn.Linear`?
:::

---

# Stacking Blocks

A real transformer is many of these blocks stacked on top of each other, each one feeding the next. The original transformer in the reading ("Attention Is All You Need") used 6 blocks, 512 numbers per word, 8 heads, and 2,048 numbers in the middle of the feed-forward part. Language models today stack many more, but the block is the same.

PyTorch has the block built in as `nn.TransformerEncoderLayer`. Print one with our sizes and you will see our pieces, plus some dropout (from regularization):

In [ ]:
# RUN THIS. PyTorch's built in block, with our sizes.
built_in = nn.TransformerEncoderLayer(
    d_model=16,            # numbers per word
    nhead=2,               # heads
    dim_feedforward=32,    # numbers in the middle of the feed-forward part
    batch_first=True)
print(built_in)

We wrote our own today so you can see every piece. Next class we stack blocks to build and train a tiny GPT.

# Why Does This Work? The Honest Answer

You might be feeling a bit uneasy right now. We built every piece of this by hand, and it works, but WHY does a stack of these blocks end up "understanding" language? Here is the honest answer, in three levels:

1. **One neuron.** A weighted sum and a nonlinearity. We understand this completely.
2. **Learning.** Gradient descent nudges every weight a tiny bit to make the loss smaller. We understand this completely too.
3. **What the trained network ends up doing.** This is the part nobody fully understands yet! It is a whole research area called **interpretability**.

So we understand the *mechanism* completely, but not always the *solution* the network finds. That is okay, and it is pretty normal in science. Think about evolution: one simple rule (keep what helps you survive), repeated over and over, builds an eye that nobody designed. Gradient descent is one simple rule (move toward a smaller loss), repeated millions of times.


What researches find in real models is that some heads learn jobs we can name: one looks at the word right before it, one connects a verb to its object, one connects "she" to the person she is (Clark et al. 2019, "What Does BERT Look At?"). Some heads team up to copy patterns, like predicting "B" after seeing "A" again when "A B" showed up earlier (Olsson et al. 2022, "In-context Learning and Induction Heads"). And many heads do things nobody has a name for, and a lot of them can be removed without hurting the model much (Michel et al. 2019, "Are Sixteen Heads Really Better than One?").

BEWARE: the attention weights are a clue about what a model is doing, not a full explanation (Jain and Wallace 2019, "Attention is not Explanation").

:::{.callout-note icon=false}
## Q6. Write this one out by hand

**a.** We understand neurons and gradient descent completely. What part of a trained network do we NOT fully understand?

**b.** Name one job that researchers have found a real attention head doing. Why is it okay that many heads do jobs nobody can name?
:::

---

# Putting It Together

## What this looks like in real life - The Whole Process in One Cell

Here is everything in one cell, with no helper function. The comments show the places you could change things.

In [ ]:
# LEARN TO WRITE THIS.
# A transformer review classifier from start to finish.
# It uses make_reviews and vocab_size from above.

# 1. The data. Here you could change how many reviews there are and how long they are
n_words = 20
X_train, y_train = make_reviews(4000, n_words=n_words, seed=0)
X_valid, y_valid = make_reviews(1000, n_words=n_words, seed=1)

# 2. One transformer block. Here you could change the number of heads or the size in the middle
class TransformerBlock(nn.Module):
    def __init__(self, n_numbers=16, n_heads=2):
        super().__init__()
        self.attention = nn.MultiheadAttention(n_numbers, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(n_numbers)
        self.feed_forward = nn.Sequential(
            nn.Linear(n_numbers, 2 * n_numbers),
            nn.ReLU(),
            nn.Linear(2 * n_numbers, n_numbers))
        self.norm2 = nn.LayerNorm(n_numbers)

    def forward(self, X):
        attention_out, weights = self.attention(X, X, X)   # check in with the other words
        Z = self.norm1(X + attention_out)                   # remember who I am, and do not get lost in the sauce
        return self.norm2(Z + self.feed_forward(Z))         # self reflection, remember, normalize again

# 3. The model. Here you could change the 16 numbers per word, or stack more blocks
class ReviewTransformer(nn.Module):
    def __init__(self, n_words):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 16)           # 16 numbers for each word
        self.position_embed = nn.Embedding(n_words, 16)     # 16 numbers for each spot
        self.block = TransformerBlock(16, n_heads=2)
        self.out = nn.Linear(16, 2)                         # 2 scores: negative, positive

    def forward(self, X):
        vectors = self.embed(X) + self.position_embed(torch.arange(X.shape[1]))   # word numbers + spot numbers
        Z = self.block(vectors)                             # (reviews, words, 16)
        return self.out(Z.mean(dim=1))                      # average over the words, then 2 scores

torch.manual_seed(42)
model = ReviewTransformer(n_words)

# 4. The batches, the loss and the optimizer. Here you could change the batch size or the learning rate
torch.manual_seed(42)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 5. The training loop. Here you could change the number of epochs
n_epochs = 10
for epoch in range(n_epochs):
    model.train()                             # training mode
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                 # clear the old gradients
        scores = model(X_batch)               # forward pass: 2 scores per review
        loss = loss_fn(scores, y_batch)       # how far off are we?
        loss.backward()                       # backward pass
        optimizer.step()                      # one step of Adam

    # 6. After every epoch, check the validation accuracy
    model.eval()                              # prediction mode
    with torch.no_grad():
        y_pred = model(X_valid).argmax(dim=1)
        accuracy = (y_pred == y_valid).float().mean().item()
    print("epoch", epoch + 1, "  valid accuracy:", round(accuracy, 4))

Everything from today, in one list:

1. **Multi-head attention** runs several attention heads side by side. Each has its own queries, keys and values, and the outputs go side by side into one `nn.Linear`.
2. Attention alone does not know the order of the words. A **positional embedding** adds a learned row of numbers for each spot.
3. A **residual connection** adds the input back in after attention and after the feed-forward part, like ResNet.
4. **Layer norm** keeps each word's numbers a good size, one word at a time.
5. The **feed-forward** part is a small MLP that works on each word on its own.
6. A block has the same shape in and out, so blocks stack.

# New Commands Today

| Command | What it does | The thing that trips people up |
|---|---|---|
| `nn.MultiheadAttention(16, 2, batch_first=True)` | several attention heads side by side | the number per word (16) has to divide evenly by the number of heads (2) |
| `output, weights = attention(X, X, X)` | self-attention: the words are the queries, keys and values | it gives back TWO things, and you send X in three times |
| `average_attn_weights=False` | gives back every head's table of weights | without it you get the average over the heads |
| `nn.Embedding(n_words, 16)` for spots, with `torch.arange(n_words)` | a learned row of numbers for every spot | it has to be the same size as the word embedding so you can add them |
| `nn.LayerNorm(16)` | centers and scales each word's 16 numbers | it works on one word at a time, not on the batch |
| `Z.mean(dim=1)` | the average over the words | `dim=1` is the words, `dim=0` would be the reviews |
| `nn.TransformerEncoderLayer` | PyTorch's built in block | it has dropout inside, so `model.train()` and `model.eval()` matter |

:::{.callout-tip icon=false}
## You Try: optional code

Nothing here is collected. Work through it if you want the idea to stick.

**1.** Change `n_heads=2` to `n_heads=4` in `ReviewTransformer`. Does it learn faster or slower? What happens if you try `n_heads=3`?

**2.** Stack two blocks: add `self.block2 = TransformerBlock(16, n_heads=2)` and send `Z` through it in `forward`. Does it help on these reviews?

**3.** Swap our `TransformerBlock` for `nn.TransformerEncoderLayer(d_model=16, nhead=2, dim_feedforward=32, batch_first=True)`. Do you get about the same accuracy?
:::

# Before Next Class

1. In your lecture notes notebook, add your handwritten notes and answers to the questions.
2. Do the practice problems in `HW_day17.ipynb`. They are practice, not graded, and the solutions are in the same folder.
3. Check the class website for what is due this week.
4. Coming up next: we stack these blocks and build and train a tiny GPT.

# Answers to the Q Boxes

Try every one of these by hand first. These are short summaries, not full answers, and the writing out is the part that does the work.

:::{.callout-note collapse="true"}
## Q1. Multi-head attention

**a.** Each head makes its own table of weights, so each one can look for something different. In "my friends loved the movie", one head could connect "loved" to "friends" (who loved it) while another connects "loved" to "movie" (what they loved). One head has only one table, so it would have to mix both into one set of weights.

**b.** $512 / 8 = 64$ numbers per head. The 8 outputs of 64 go side by side, so 512 numbers come out for each word, the same as went in.
:::

:::{.callout-note collapse="true"}
## Q2. Positions

**a.** Attention compares every word to every other word, and nothing in the formula says where a word is. Both reviews have the same words, so attention makes the same outputs, only in a different order, and the mean over the words is the same.

**b.** We **add** the spot numbers to the word numbers, number by number, and you can only add two lists of the same length.
:::

:::{.callout-note collapse="true"}
## Q3. Residual connections and layer norm

**a.** Then $Z = \text{LayerNorm}(X)$: the words pass through with only the layer norm. If a block has nothing useful to add, the words still pass through it, and the gradient has a short path back through the add, so it does not vanish through many blocks.

**b.** The numbers of one word: it takes the mean and standard deviation of that word's 16 numbers. It does not need any other review.
:::

:::{.callout-note collapse="true"}
## Q4. Attention and feed-forward

**a.** Attention looks at all of the words and mixes information between them. The feed-forward network looks at one word at a time and only at that word's numbers.

**b.** So the output of one block can be the input of the next one. That is what lets us stack 6 blocks, or many more.
:::

:::{.callout-note collapse="true"}
## Q5. Positions and the mean

**a.** No. On the one feeling word reviews only **which** feeling word is in the review matters, not where it is, and the model without positions can see which words are there. And the feeling word is one attention step away from every other word, so the gradient does not vanish on the way back. (Try it in the practice problems!)

**b.** The block gives back 16 numbers for every word, 20 lists per review. `nn.Linear(16, 2)` needs one list of 16 per review to make one pair of scores, so we average the 20 lists into one.
:::

:::{.callout-note collapse="true"}
## Q6. Why does this work?

**a.** What the trained network ends up doing, the solution it finds. We know exactly how each neuron computes and how gradient descent changes the weights, but we can not always say in words what job each head or neuron learned. That is the research area called interpretability.

**b.** For example, looking at the word right before it, connecting a verb to its object, or connecting "she" to the person she is. The loss only rewards good predictions, it does not care whether a person can name what each head does. Any set of weights that predicts well is a good solution to gradient descent.
:::